# 05 · Model possession (H vs A) dari pseudo-label

**Tujuan.** Mengubah fitur `04` menjadi satu angka per frame: **log-odds bahwa `H` yang menguasai bola**
(positif = H, negatif = A), yang bekerja **juga saat bola tidak terlihat**.

| | |
|---|---|
| **Input** | `cache/features_ha.npz`, `cache/dead.npz` |
| **Output** | `cache/possession_model.npz`, `cache/possession_model.json`, laporan `cache/reports/05_model_possession.*` |
| **Waktu** | ± 30 s |

### Ide: belajar dari momen yang jelas, menerapkan ke momen yang tidak jelas

GT publik terlalu kecil untuk melatih (17 possession), dan GT penuh privat. Tetapi pertandingan ini sendiri penuh
momen di mana possession **jelas tanpa label**: bola terlihat, satu pemain menempel padanya, dan tak ada lawan di
dekatnya selama setengah detik. Itulah **pseudo-label** — dibuat oleh aturan, dari seluruh match, tanpa GT.

Pada momen-momen itu kita mengamati **bentuk kedua tim**. Model bentuk-tim yang dilatih di sana lalu dipakai di
semua frame — termasuk ± 41% frame dinilai tanpa bola.

1. Pseudo-label "kontrol jelas" (bagian 1).
2. Model logistik pada **fitur bentuk tim saja** (tanpa fitur bola — supaya tidak sekadar menghafal aturan
   pseudo-label, dan supaya berlaku saat bola hilang) (bagian 2).
3. Validasi silang **per blok waktu** (blok yang diuji tidak pernah ikut dilatih) (bagian 3).
4. Bobot bukti "pemain terdekat" diperkirakan **tanpa GT**: dari frame di mana model bentuk-tim yakin, seberapa
   sering pemain terdekat berasal dari tim yang sama (bagian 4).
5. Log-odds gabungan = bentuk tim + bobot × pemain terdekat (bagian 5).

In [ ]:
%run ./common.ipynb

## Parameter

| Parameter | Nilai | Alasan |
|---|---|---|
| `CTRL_R` | 1,5 m | pemain "menempel" bola (jangkauan menggiring/menerima) |
| `FREE_R` | 3,0 m | tak ada lawan dalam jarak ini → belum direbut |
| `CTRL_MIN` | 12 frame (0,5 s) | kontrol harus bertahan, bukan sentuhan sesaat (CHALLENGE.md: "controlled") |
| `SHAPE_FEATS` | 6 selisih bentuk tim halus | fitur bentuk dari `04`, tanpa fitur bola |
| `L2` | 1,0 | regularisasi ringan (sampel puluhan ribu frame; fitur berkorelasi) |
| `N_BLOCKS` | 10 | blok waktu untuk validasi silang (± 10 menit per blok) |
| `CONF_LOGIT` | 2,0 | ambang "model bentuk yakin" (P ≥ 0,88) untuk menaksir keandalan pemain terdekat |

In [ ]:
CTRL_R = 1.5
FREE_R = 3.0
CTRL_MIN = 12
SHAPE_FEATS = ["width_diff_s", "spread_diff_s", "depth_diff_s", "speed_diff_s", "v_goal_diff_s", "advance_diff_s"]
L2 = 1.0
N_BLOCKS = 10
CONF_LOGIT = 2.0

In [ ]:
rep = Report("05_model_possession")
with np.load(CACHE / "features_ha.npz") as fz:
    F = {n: fz[n] for n in fz["names"]}
with np.load(CACHE / "dead.npz") as dz:
    d_mask = dz["d_mask"]
z = load_prep()
live = GRADED & ~d_mask
observed = z["ball_src"] == 1          # bola teramati langsung (bukan interpolasi)

## 1. Pseudo-label "kontrol jelas"

Frame *t* berlabel `H` bila: dinilai, bukan `D` menurut `03`, bola **teramati langsung**, pemain H terdekat
≤ `CTRL_R`, pemain A terdekat ≥ `FREE_R` — dan kondisi itu bertahan ≥ `CTRL_MIN` frame berturut-turut. Sama untuk `A`.

Hasilnya sengaja sedikit tetapi bersih. Kemurniannya dicek terhadap GT publik — **hanya sebagai informasi**:
aturan ini tidak disetel di sana.

In [ ]:
def control_runs(d_own, d_other):
    cond = live & observed & (d_own <= CTRL_R) & (d_other >= FREE_R)
    cond = np.nan_to_num(cond, nan=False).astype(bool)
    out = np.zeros(N_FRAMES, dtype=bool)
    n = 0
    for a, b in zip(*runs(cond)):
        if b - a >= CTRL_MIN:
            out[a:b] = True
            n += 1
    return out, n


ctrl_H, n_runs_H = control_runs(F["d_H"], F["d_A"])
ctrl_A, n_runs_A = control_runs(F["d_A"], F["d_H"])
assert not (ctrl_H & ctrl_A).any()
pseudo = np.zeros(N_FRAMES, dtype=np.int8)          # +1 H, −1 A, 0 tak berlabel
pseudo[ctrl_H], pseudo[ctrl_A] = 1, -1
labeled = pseudo != 0

rep("pseudo-label: H {} frame ({} momen) | A {} frame ({} momen) | {:.1f}% frame hidup".format(
    int(ctrl_H.sum()), n_runs_H, int(ctrl_A.sum()), n_runs_A, 100 * labeled[live].mean()))
for h in (1, 2):
    m = labeled & (HALF == h)
    rep("  babak {}: {} frame, porsi H {:.2f}".format(h, int(m.sum()), float((pseudo[m] == 1).mean())))

gt = pm.read_label_csv(str(GT_PUBLIC))["label"].astype(str).str.strip().str.upper().to_numpy()
gt_code = np.select([gt == "H", gt == "A"], [1, -1], 0)
pub = labeled & (gt_code != 0)
rep("informasi — kemurnian pseudo-label di jendela publik: {:.1f}% dari {} frame; {} frame jatuh di GT D".format(
    100 * float((pseudo[pub] == gt_code[pub]).mean()), int(pub.sum()), int((labeled & (gt == "D")).sum())))

## 2. Model logistik bentuk tim

Fitur distandarkan dengan rata-rata/simpangan di frame dinilai; `NaN` → 0 (netral). Kelas diseimbangkan dengan
bobot sampel (porsi H dan A pseudo-label tidak harus sama dengan porsi possession sebenarnya). Pelatihan memakai
Newton-Raphson (IRLS) dengan regularisasi L2 — tanpa scikit-learn, deterministik.

Karena frame berurutan dalam satu momen kontrol hampir identik, setiap **momen** diberi bobot total yang sama
(bobot frame = 1 / panjang momen). Tanpa ini, satu giringan panjang akan mendominasi.

In [ ]:
X_raw = np.stack([F[n] for n in SHAPE_FEATS], axis=1).astype(np.float64)
mu = np.nanmean(X_raw[GRADED], axis=0)
sd = np.nanstd(X_raw[GRADED], axis=0) + 1e-9
X = (X_raw - mu) / sd
X[~np.isfinite(X)] = 0.0

run_w = np.zeros(N_FRAMES)
for mask in (ctrl_H, ctrl_A):
    for a, b in zip(*runs(mask)):
        run_w[a:b] = 1.0 / (b - a)


def fit_logit(Xs, y, w, lam=L2, iters=50):
    n, d = Xs.shape
    pos = w[y == 1].sum()
    neg = w[y == 0].sum()
    sw = w * np.where(y == 1, 0.5 / pos, 0.5 / neg) * n          # kelas seimbang, skala ± n
    Xb = np.c_[np.ones(n), Xs]
    beta = np.zeros(d + 1)
    R = lam * np.eye(d + 1)
    R[0, 0] = 0.0
    for _ in range(iters):
        p = 1.0 / (1.0 + np.exp(-np.clip(Xb @ beta, -30, 30)))
        g = Xb.T @ (sw * (y - p)) - R @ beta
        H = Xb.T @ (Xb * (sw * p * (1 - p) + 1e-12)[:, None]) + R
        step = np.linalg.solve(H, g)
        beta += step
        if np.abs(step).max() < 1e-8:
            break
    return beta


def logit(Xs, beta):
    return beta[0] + Xs @ beta[1:]


y_all = (pseudo == 1).astype(float)
beta = fit_logit(X[labeled], y_all[labeled], run_w[labeled])
coef = pd.Series(beta[1:], index=SHAPE_FEATS).round(3)
rep("intersep {:+.3f}".format(beta[0]))
rep.table(coef.to_frame("koefisien (fitur terstandar)"), "model bentuk tim (seluruh pseudo-label):")

## 3. Validasi silang per blok waktu

Frame dinilai dibagi menjadi `N_BLOCKS` blok berurutan. Untuk setiap blok: latih pada pseudo-label di luar blok,
prediksi **semua** frame di dalam blok. Hasilnya `L_shape_oof` — prediksi yang tidak pernah melihat label di
sekitarnya — dipakai untuk semua evaluasi di bawah dan di `06`.

Metrik pada pseudo-label yang disisihkan: AUC dan akurasi tanda. Frame pseudo-label adalah momen yang *mudah*,
jadi angka ini batas atas; angka GT publik (informasi) menunjukkan perilaku di semua frame hidup.

In [ ]:
graded_idx = np.flatnonzero(GRADED)
edges = np.linspace(0, len(graded_idx), N_BLOCKS + 1).astype(int)
block = np.full(N_FRAMES, -1)
for k in range(N_BLOCKS):
    block[graded_idx[edges[k]:edges[k + 1]]] = k

L_oof = np.zeros(N_FRAMES)
fold_rows = []
for k in range(N_BLOCKS):
    tr = labeled & (block != k) & (block >= 0)
    b_k = fit_logit(X[tr], y_all[tr], run_w[tr])
    te = block == k
    L_oof[te] = logit(X[te], b_k)
    held = te & labeled
    fold_rows.append({"blok": k, "babak": int(np.bincount(HALF[te]).argmax()), "pseudo uji": int(held.sum()),
                      "AUC": round(auc(L_oof[held & (pseudo == 1)], L_oof[held & (pseudo == -1)]), 3),
                      "akurasi": round(float((np.sign(L_oof[held]) == pseudo[held]).mean()), 3)})
folds = pd.DataFrame(fold_rows).set_index("blok")
rep.table(folds, "validasi silang (pseudo-label disisihkan):")
rep("rata-rata: AUC {:.3f} | akurasi {:.3f} | blok terburuk AUC {:.3f}".format(
    folds.AUC.mean(), folds.akurasi.mean(), folds.AUC.min()))

## 4. Seberapa bisa dipercaya "pemain terdekat"? — ditaksir tanpa GT

Di frame hidup dengan bola terlihat **dan** model bentuk-tim yakin (|L| ≥ `CONF_LOGIT`, out-of-fold): seberapa sering
pemain terdekat berasal dari tim yang ditunjuk model? Bila keduanya kerap sepakat, pemain terdekat informatif.
Kesepakatan itu (*q*) menjadi bobot log-odds per frame untuk pemain terdekat: **β = log(q / (1 − q))**.

Ini taksiran kasar (model bentuk sendiri tidak sempurna), tetapi seluruhnya berasal dari data pertandingan, bukan GT.

In [ ]:
near = F["near"].astype(int)
conf = live & (near != 0) & (np.abs(L_oof) >= CONF_LOGIT)
q = float((near[conf] == np.sign(L_oof[conf])).mean())
BETA_NEAR = float(np.log(q / (1 - q)))
rep("frame yakin & bola terlihat: {} | kesepakatan q = {:.3f} → β_near = {:.3f}".format(int(conf.sum()), q, BETA_NEAR))

## 5. Log-odds gabungan & cek

`L_total = L_shape + β_near × near`

- `L_shape` untuk pipeline = prediksi **out-of-fold** (konsisten dengan evaluasi; tidak ada frame yang dinilai oleh
  model yang dilatih pada momen di sekitarnya).
- Di frame `D` nilainya tetap dihitung tetapi tidak dipakai (decoder memaksa `D`).

Informasi GT publik: AUC dan akurasi tanda untuk H vs A di frame hidup GT — semua frame, dan khusus saat bola hilang.

In [ ]:
L_near = BETA_NEAR * near
L_total = L_oof + L_near

isH, isA = gt == "H", gt == "A"
hidden = F["near"] == 0
info = []
for name, L in (("bentuk tim (oof)", L_oof), ("pemain terdekat", L_near.astype(float)), ("gabungan", L_total)):
    row = {"sumber": name, "AUC semua": round(auc(L[isH], L[isA]), 3),
           "akurasi tanda": round(float((np.sign(L[isH | isA]) == gt_code[isH | isA]).mean()), 3)}
    row["AUC bola hilang"] = round(auc(L[isH & hidden], L[isA & hidden]), 3)
    info.append(row)
info = pd.DataFrame(info).set_index("sumber")
rep.table(info, "informasi — GT publik, frame hidup:")
rep("seluruh match, frame hidup: P(H) rata-rata {:.3f} | porsi frame L_total > 0: {:.3f}".format(
    float((1 / (1 + np.exp(-L_total[live]))).mean()), float((L_total[live] > 0).mean())))

In [ ]:
np.savez(CACHE / "possession_model.npz", L_shape=L_oof.astype(np.float32), L_near=L_near.astype(np.float32),
         L_total=L_total.astype(np.float32), pseudo=pseudo)
(CACHE / "possession_model.json").write_text(json.dumps({
    "shape_feats": SHAPE_FEATS, "mu": mu.tolist(), "sd": sd.tolist(), "beta_full": beta.tolist(),
    "beta_near": BETA_NEAR, "q_near": q, "params": {"CTRL_R": CTRL_R, "FREE_R": FREE_R, "CTRL_MIN": CTRL_MIN,
                                                    "L2": L2, "N_BLOCKS": N_BLOCKS, "CONF_LOGIT": CONF_LOGIT}},
    indent=2), encoding="utf-8")
rep.save({"coef": coef.to_dict(), "folds": folds.reset_index().to_dict("records"), "q_near": q,
          "beta_near": BETA_NEAR, "public_info": info.reset_index().to_dict("records")})

## Ringkasan & keputusan

| | |
|---|---|
| Pseudo-label "kontrol jelas" | ± 18 rb frame, ± 570 momen (± 22% frame hidup); kemurnian di jendela publik ± 93% (informasi) |
| Model bentuk tim, validasi silang per blok | AUC rata-rata ± 0,91 (blok terburuk ± 0,79), akurasi ± 0,86 |
| Keandalan pemain terdekat (tanpa GT) | q ± 0,73 → β_near ± 1,0 |
| GT publik (informasi, out-of-fold) | bentuk tim: AUC ± 0,98, akurasi tanda ± 0,92 — **juga saat bola hilang**; pemain terdekat saja: ± 0,73 |

**Bacaan**

1. **Pendekatan pseudo-label berhasil.** Model yang tidak pernah melihat GT memisahkan H dan A di 5 menit publik
   hampir sempurna, dan tetap bekerja saat bola tidak terlihat — persis celah yang tak bisa ditutup baseline.
2. **Taksiran keandalan pemain terdekat tanpa GT (q ± 0,73) cocok dengan yang terlihat di GT publik** (kemurnian
   ± 74% di `02`) — konfirmasi independen bahwa model bentuk-tim menjadi "wasit" yang masuk akal.
3. **Arah koefisien** sesuai temuan `02`: tim menguasai lebih menyebar/lebar dan lebih lambat. Kedalaman bertanda
   negatif karena berkorelasi tinggi dengan sebaran — keduanya harus dibaca bersama.
4. **Gabungan ≈ bentuk tim saja** di jendela publik; pemain terdekat menambah sedikit, tetapi penting di saat
   pergantian (bentuk tim berubah **lambat** setelah bola direbut — lihat di bawah).

**Keterbatasan yang dibawa ke tahap berikut**

- **Bentuk tim tertinggal waktu.** Setelah bola direbut, tim butuh beberapa detik untuk melebar. Bukti bentuk
  akan menggeser batas possession **terlambat**. → Fase 2.7: batas diperhalus dengan bukti bola (sentuhan/kontrol).
- Pseudo-label condong ke H (± 64%); diatasi dengan bobot kelas seimbang, tetapi intersep tetap dipantau.
- Dua blok waktu lebih lemah (AUC ± 0,79): kandidat diselidiki (fase permainan tertentu? kualitas tracking?).